# 2025 Strength of Schedule: Opponent-Adjusted EPA

For each NFL team, compute two SOS scores based on their 2025 opponents' 2025 performance:
- **Offensive SOS** â€” average adjusted defensive EPA of opponents (lower = harder for your offense)
- **Defensive SOS** â€” average adjusted offensive EPA of opponents (higher = harder for your defense)

EPA values are opponent-adjusted: each team's unit efficiency is credited/discounted based on the quality of opponents they faced in 2025.

In [1]:
# Run this cell only if packages aren't installed yet
# !pip install nfl_data_py pandas -q

In [2]:
import nfl_data_py as nfl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Step 1: Load 2025 Play-by-Play Data

Pull all 2025 plays (regular season only). We filter to standard pass and run plays only — no kickoffs, two-point conversions, or special teams.

In [3]:
pbp = nfl.import_pbp_data([2025])

# Filter to pass and run plays, exclude two-point attempts and preseason
pbp_filtered = pbp[
    (pbp['season_type'] == 'REG') &
    (pbp['play_type'].isin(['pass', 'run'])) &
    (pbp['two_point_attempt'] == 0) &
    (pbp['epa'].notna()) &
    (pbp['posteam'].notna())
].copy()

print(f"Total plays: {len(pbp_filtered):,}")
print(f"Weeks covered: {pbp_filtered['week'].min()} to {pbp_filtered['week'].max()}")
print(f"Teams: {pbp_filtered['posteam'].nunique()}")

2025 done.
Downcasting floats.


Total plays: 32,814
Weeks covered: 1 to 18
Teams: 32


## Step 2: Compute Raw EPA/Play by Unit

- **Offensive EPA/play**: from the offense's perspective (`posteam`)
- **Defensive EPA/play**: from the defense's perspective (`defteam`) â€” lower is better for the defense

In [4]:
raw_off_epa = (
    pbp_filtered.groupby('posteam')['epa']
    .mean()
    .rename('raw_off_epa')
)

raw_def_epa = (
    pbp_filtered.groupby('defteam')['epa']
    .mean()
    .rename('raw_def_epa')
)

team_epa = pd.concat([raw_off_epa, raw_def_epa], axis=1).reset_index()
team_epa.columns = ['team', 'raw_off_epa', 'raw_def_epa']

print(f"League avg offensive EPA/play: {team_epa['raw_off_epa'].mean():.4f}")
print(f"League avg defensive EPA/play: {team_epa['raw_def_epa'].mean():.4f}")
team_epa.sort_values('raw_off_epa', ascending=False)

League avg offensive EPA/play: 0.0058
League avg defensive EPA/play: 0.0068


,team,raw_off_epa,raw_def_epa
21,NE,0.1566,-0.0464
16,LA,0.1448,-0.0624
3,BUF,0.1410,-0.0146
11,GB,0.1130,0.0331
8,DAL,0.0952,0.1670
28,SF,0.0891,0.0745
10,DET,0.0813,-0.0094
5,CHI,0.0796,0.0169
13,IND,0.0752,0.0014
9,DEN,0.0478,-0.0863


## Step 3: Build Game-Level Opponent Lookup

For each team-game, record who their opponent was. We'll use this to compute the average opponent unit EPA each team faced.

In [5]:
# Get unique games with home/away teams
games_2025 = (
    pbp_filtered[['game_id', 'posteam', 'defteam']]
    .drop_duplicates()
)

# Build a lookup: for each (team, game), who was the opponent?
# From posteam perspective: posteam faced defteam's defense
off_matchups = games_2025[['posteam', 'defteam']].copy()
off_matchups.columns = ['team', 'opponent']

# From defteam perspective: defteam faced posteam's offense
def_matchups = games_2025[['defteam', 'posteam']].copy()
def_matchups.columns = ['team', 'opponent']

off_matchups = off_matchups.drop_duplicates()
def_matchups = def_matchups.drop_duplicates()

print(f"Offense matchup rows: {len(off_matchups)}")
print(f"Defense matchup rows: {len(def_matchups)}")

Offense matchup rows: 448
Defense matchup rows: 448


## Step 4: Opponent-Adjust EPA

**Formula:**
```
adj_off_epa = raw_off_epa - (avg_opp_def_epa - league_avg_def_epa)
adj_def_epa = raw_def_epa - (avg_opp_off_epa - league_avg_off_epa)
```

If your offense faced defenses that were better than average (lower raw_def_epa), we credit your offense upward. If your defense faced offenses that were worse than average, we penalize your defense downward.

In [6]:
league_avg_off = team_epa['raw_off_epa'].mean()
league_avg_def = team_epa['raw_def_epa'].mean()

# Average defensive EPA of opponents each team's offense faced
off_matchups_epa = off_matchups.merge(
    team_epa[['team', 'raw_def_epa']],
    left_on='opponent', right_on='team'
).groupby('team_x')['raw_def_epa'].mean().rename('avg_opp_def_epa')

# Average offensive EPA of opponents each team's defense faced
def_matchups_epa = def_matchups.merge(
    team_epa[['team', 'raw_off_epa']],
    left_on='opponent', right_on='team'
).groupby('team_x')['raw_off_epa'].mean().rename('avg_opp_off_epa')

team_epa = team_epa.set_index('team')
team_epa = team_epa.join(off_matchups_epa).join(def_matchups_epa)

team_epa['adj_off_epa'] = team_epa['raw_off_epa'] - (team_epa['avg_opp_def_epa'] - league_avg_def)
team_epa['adj_def_epa'] = team_epa['raw_def_epa'] - (team_epa['avg_opp_off_epa'] - league_avg_off)

team_epa = team_epa.reset_index()
print("Adjusted EPA by team:")
team_epa[['team', 'raw_off_epa', 'adj_off_epa', 'raw_def_epa', 'adj_def_epa']].sort_values('adj_off_epa', ascending=False)

Adjusted EPA by team:


,team,raw_off_epa,adj_off_epa,raw_def_epa,adj_def_epa
16,LA,0.1448,0.1607,-0.0624,-0.0589
3,BUF,0.1410,0.1436,-0.0146,0.0075
21,NE,0.1566,0.1251,-0.0464,0.0065
28,SF,0.0891,0.1099,0.0745,0.0912
13,IND,0.0752,0.0969,0.0014,0.0115
11,GB,0.1130,0.0931,0.0331,0.0361
8,DAL,0.0952,0.0858,0.1670,0.1829
10,DET,0.0813,0.0682,-0.0094,-0.0214
5,CHI,0.0796,0.0608,0.0169,0.0323
2,BAL,0.0339,0.0439,0.0278,0.0119


## Step 5: Load 2025 Schedule

Pull the full 2026 schedule (regular season only for now â€” playoffs aren't scheduled yet).

In [7]:
schedule_2025 = nfl.import_schedules([2026])

# Keep regular season only
schedule_2025 = schedule_2025[schedule_2025['game_type'] == 'REG']

print(f"2025 regular season games: {len(schedule_2025)}")
print(f"Weeks: {schedule_2025['week'].min()} to {schedule_2025['week'].max()}")
schedule_2025[['week', 'home_team', 'away_team']].head(10)

2025 regular season games: 272
Weeks: 1 to 18


,week,home_team,away_team
7276,1,SEA,NE
7277,1,LA,SF
7278,1,CAR,CHI
7279,1,CIN,TB
7280,1,DET,NO
7281,1,HOU,BUF
7282,1,IND,BAL
7283,1,JAX,CLE
7284,1,PIT,ATL
7285,1,TEN,NYJ


## Step 6: Compute 2025 SOS Scores

For each team's 2025 opponents:
- **Offensive SOS** = average opponent `adj_def_epa` (lower = tougher schedule for your offense)
- **Defensive SOS** = average opponent `adj_off_epa` (higher = tougher schedule for your defense)

In [8]:
# Build team-opponent pairs from 2026 schedule
home = schedule_2025[['home_team', 'away_team']].rename(columns={'home_team': 'team', 'away_team': 'opponent'})
away = schedule_2025[['away_team', 'home_team']].rename(columns={'away_team': 'team', 'home_team': 'opponent'})
matchups_2025 = pd.concat([home, away], ignore_index=True)

# Join with 2025 adjusted EPA
sos = matchups_2025.merge(
    team_epa[['team', 'adj_off_epa', 'adj_def_epa']],
    left_on='opponent', right_on='team',
    suffixes=('', '_opp')
)

sos_scores = sos.groupby('team').agg(
    off_sos=('adj_def_epa', 'mean'),   # avg opponent defensive EPA â€” lower = harder for your offense
    def_sos=('adj_off_epa', 'mean'),   # avg opponent offensive EPA â€” higher = harder for your defense
).reset_index()

sos_scores = sos_scores.sort_values('off_sos')
print("2025 SOS Scores (sorted by toughest offensive schedule):")
sos_scores

2025 SOS Scores (sorted by toughest offensive schedule):


,team,off_sos,def_sos
4,CAR,-0.0285,-0.0136
5,CHI,-0.0173,0.0052
26,PIT,-0.0154,-0.0191
18,LV,-0.0075,0.0019
6,CIN,-0.0056,-0.0238
29,TB,-0.0043,-0.0115
28,SF,-0.0033,0.0092
11,GB,-0.0013,0.0112
17,LAC,-0.0012,0.0106
2,BAL,0.0014,-0.0187


## Step 7: Sanity Check & Export

Quick look at the spread and export to CSV for further analysis.

In [9]:
print("Offensive SOS range:")
print(f"  Hardest (lowest): {sos_scores['off_sos'].min():.4f} â€” {sos_scores.loc[sos_scores['off_sos'].idxmin(), 'team']}")
print(f"  Easiest (highest): {sos_scores['off_sos'].max():.4f} â€” {sos_scores.loc[sos_scores['off_sos'].idxmax(), 'team']}")

print("\nDefensive SOS range:")
print(f"  Hardest (highest): {sos_scores['def_sos'].max():.4f} â€” {sos_scores.loc[sos_scores['def_sos'].idxmax(), 'team']}")
print(f"  Easiest (lowest): {sos_scores['def_sos'].min():.4f} â€” {sos_scores.loc[sos_scores['def_sos'].idxmin(), 'team']}")

sos_scores.to_csv('sos_2025.csv', index=False)
print("\nExported to sos_2025.csv")

Offensive SOS range:
  Hardest (lowest): -0.0285 â€” CAR
  Easiest (highest): 0.0444 â€” PHI

Defensive SOS range:
  Hardest (highest): 0.0374 â€” SEA
  Easiest (lowest): -0.0238 â€” CIN

Exported to sos_2025.csv


## Step 8: Top 10 Opponents by EPA

Rank the toughest opponents a team could face â€” top 10 offenses (hardest on opposing defenses) and top 10 defenses (hardest on opposing offenses).

In [10]:
print("=== Top 10 Offenses (adj EPA/play) ===")
top_off = team_epa[['team', 'adj_off_epa']].sort_values('adj_off_epa', ascending=False).head(10).reset_index(drop=True)
top_off.index += 1
print(top_off.to_string())

print("\n=== Top 10 Defenses (adj EPA/play allowed) ===")
top_def = team_epa[['team', 'adj_def_epa']].sort_values('adj_def_epa').head(10).reset_index(drop=True)
top_def.index += 1
print(top_def.to_string())

=== Top 10 Offenses (adj EPA/play) ===
   team  adj_off_epa
1    LA       0.1607
2   BUF       0.1436
3    NE       0.1251
4    SF       0.1099
5   IND       0.0969
6    GB       0.0931
7   DAL       0.0858
8   DET       0.0682
9   CHI       0.0608
10  BAL       0.0439

=== Top 10 Defenses (adj EPA/play allowed) ===
   team  adj_def_epa
1   HOU      -0.1403
2   MIN      -0.1132
3   SEA      -0.1076
4   PHI      -0.1025
5   CLE      -0.0934
6   JAX      -0.0855
7    NO      -0.0742
8   DEN      -0.0695
9   LAC      -0.0670
10   LA      -0.0589


## Step 9: Regenerate Charts

Write updated PNG charts to `public/images/sos-2026/` matching the site color palette.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG     = "#0a0a0a"
FG     = "#e5e5e5"
HARD   = "#ef4444"
EASY   = "#22d3ee"
ORANGE = "#f97316"
OUT    = r"C:\Users\Arjun\OneDrive\Documents\nfl_tempo\nfl_tempo\public\images\sos-2026"

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=FG, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")

# Offensive SOS bar chart
off_sorted = sos_scores.sort_values("off_sos", ascending=False)
colors_off = [HARD if v < off_sorted["off_sos"].median() else EASY for v in off_sorted["off_sos"]]
fig, ax = plt.subplots(figsize=(10, 10), facecolor=BG)
style_ax(ax)
ax.barh(off_sorted["team"], off_sorted["off_sos"], color=colors_off)
ax.axvline(off_sorted["off_sos"].median(), color=FG, linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Avg Opponent Defensive EPA/play (lower = harder)", color=FG, fontsize=9)
ax.set_title("2026 Offensive Strength of Schedule", color=FG, fontsize=13, fontweight="bold", pad=12)
hard_p = mpatches.Patch(color=HARD, label="Harder schedule")
easy_p = mpatches.Patch(color=EASY, label="Easier schedule")
ax.legend(handles=[hard_p, easy_p], facecolor="#111111", edgecolor="#333333", labelcolor=FG, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/off-sos-2026.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved off-sos-2026.png")

# Defensive SOS bar chart
def_sorted = sos_scores.sort_values("def_sos")
colors_def = [HARD if v > def_sorted["def_sos"].median() else EASY for v in def_sorted["def_sos"]]
fig, ax = plt.subplots(figsize=(10, 10), facecolor=BG)
style_ax(ax)
ax.barh(def_sorted["team"], def_sorted["def_sos"], color=colors_def)
ax.axvline(def_sorted["def_sos"].median(), color=FG, linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Avg Opponent Offensive EPA/play (higher = harder)", color=FG, fontsize=9)
ax.set_title("2026 Defensive Strength of Schedule", color=FG, fontsize=13, fontweight="bold", pad=12)
hard_p = mpatches.Patch(color=HARD, label="Harder schedule")
easy_p = mpatches.Patch(color=EASY, label="Easier schedule")
ax.legend(handles=[hard_p, easy_p], facecolor="#111111", edgecolor="#333333", labelcolor=FG, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/def-sos-2026.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved def-sos-2026.png")

# SOS quadrant scatter — both axes higher = harder
# x = Opponent DEF Score = -off_sos (higher = tougher defenses your offense faces)
# y = Opponent OFF Score =  def_sos (higher = tougher offenses your defense faces)
sos_scores["opp_def_score"] = -sos_scores["off_sos"]
fig, ax = plt.subplots(figsize=(10, 9), facecolor=BG)
style_ax(ax)
med_x = sos_scores["opp_def_score"].median()
med_y = sos_scores["def_sos"].median()
ax.axvline(med_x, color="#333333", linewidth=1)
ax.axhline(med_y, color="#333333", linewidth=1)
ax.scatter(sos_scores["opp_def_score"], sos_scores["def_sos"], color=ORANGE, s=60, zorder=3)
for _, row in sos_scores.iterrows():
    ax.annotate(row["team"], (row["opp_def_score"], row["def_sos"]),
                textcoords="offset points", xytext=(5, 3),
                color=FG, fontsize=7)
ax.set_xlabel("Opponent DEF Score (higher = harder for your offense)", color=FG, fontsize=9)
ax.set_ylabel("Opponent OFF Score (higher = harder for your defense)", color=FG, fontsize=9)
ax.set_title("2026 SOS Quadrant", color=FG, fontsize=13, fontweight="bold", pad=12)
x_min, x_max = sos_scores["opp_def_score"].min(), sos_scores["opp_def_score"].max()
y_min, y_max = sos_scores["def_sos"].min(), sos_scores["def_sos"].max()
ax.annotate("Easy / Easy", xy=(x_min + 0.001, y_min + 0.001), color="#555555", fontsize=7)
ax.annotate("Hard / Hard", xy=(x_max - 0.010, y_max - 0.004), color="#555555", fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT}/sos-quadrant-2026.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.close()
print("Saved sos-quadrant-2026.png")
